In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

In [4]:
import sqlite3

ruta_db = Path(r"datos\intermedios\inmobiliario.sqlite")

with sqlite3.connect(ruta_db) as conn:
    listings = pd.read_sql_query("SELECT * FROM listings", conn)
    listings_det = pd.read_sql_query("SELECT * FROM listings_det", conn)
    euros_m2 = pd.read_sql_query("SELECT * FROM euros_m2", conn)

print(f"listings: {listings.shape[0]:,} filas")
print(f"listings_det: {listings_det.shape[0]:,} filas")
print(f"euros_m2: {euros_m2.shape[0]:,} filas")

conn.close()

listings: 17,070 filas
listings_det: 14,218 filas
euros_m2: 21 filas


In [10]:
listings_full = listings.merge(
    listings_det,
    on="id",
    how="inner"
)

listings_full.head()

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,availability_365,price_m2,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,10865,Perfect property if you need to come to Madrid...,2,1,1.0,2.0,5.00,28
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,7771,You'll love it. It is the ideal apartment to e...,6,1,2.0,5.0,4.98,255
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,7771,The apartment location and modern interior des...,4,1,2.0,4.0,4.95,255
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,7771,"Charming apartment with all the comforts, loca...",4,1,1.0,2.0,4.93,255
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,7771,Non-touristic stays only · Seasonal rental (ar...,6,2,3.0,4.0,4.97,255


In [18]:
listings_full = listings_full.rename(columns={'price': 'price_per_night'})
listings_full.head()

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price_per_night,minimum_nights,availability_365,price_m2,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,10865,Perfect property if you need to come to Madrid...,2,1,1.0,2.0,5.00,28
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,7771,You'll love it. It is the ideal apartment to e...,6,1,2.0,5.0,4.98,255
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,7771,The apartment location and modern interior des...,4,1,2.0,4.0,4.95,255
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,7771,"Charming apartment with all the comforts, loca...",4,1,1.0,2.0,4.93,255
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,7771,Non-touristic stays only · Seasonal rental (ar...,6,2,3.0,4.0,4.97,255


In [19]:
listings_full.room_type.unique()

<StringArray>
['Entire home/apt', 'Private room', 'Shared room']
Length: 3, dtype: str

In [30]:
room_type = (
    listings_full["room_type"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

listings_full["total_night_price"] = np.select(
    [
        room_type.eq("Entire home/apt"),
        room_type.eq("Private room"),
        room_type.eq("Shared room"),
    ],
    [
        listings_full["price_per_night"],
        listings_full["price_per_night"] * listings_full["bedrooms"],
        listings_full["price_per_night"] * listings_full["accommodates"],
    ],
    default=np.nan,
)

listings_full.head()

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price_per_night,minimum_nights,availability_365,price_m2,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,total_night_price
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,10865,Perfect property if you need to come to Madrid...,2,1,1.0,2.0,5.00,28,387.0
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,7771,You'll love it. It is the ideal apartment to e...,6,1,2.0,5.0,4.98,255,414.0
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,7771,The apartment location and modern interior des...,4,1,2.0,4.0,4.95,255,213.0
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,7771,"Charming apartment with all the comforts, loca...",4,1,1.0,2.0,4.93,255,166.0
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,7771,Non-touristic stays only · Seasonal rental (ar...,6,2,3.0,4.0,4.97,255,204.0


In [32]:
listings_full["anual_income"] = (
    listings_full["price_per_night"]
    * listings_full["estimated_occupancy_l365d"]
)
listings_full.head()

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price_per_night,minimum_nights,availability_365,...,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,total_night_price,ingreso anual,anual_income
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,...,Perfect property if you need to come to Madrid...,2,1,1.0,2.0,5.00,28,387.0,10836.0,10836.0
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,...,You'll love it. It is the ideal apartment to e...,6,1,2.0,5.0,4.98,255,414.0,105570.0,105570.0
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,...,The apartment location and modern interior des...,4,1,2.0,4.0,4.95,255,213.0,54315.0,54315.0
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,...,"Charming apartment with all the comforts, loca...",4,1,1.0,2.0,4.93,255,166.0,42330.0,42330.0
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,...,Non-touristic stays only · Seasonal rental (ar...,6,2,3.0,4.0,4.97,255,204.0,52020.0,52020.0


In [41]:
listings_full.bedrooms.unique()

array([1., 2., 3., 4., 5., 8., 6.])

In [42]:

pd.crosstab(
    listings_full["bedrooms"],
    listings_full["bathrooms"],
    margins=True,
    margins_name="Total",
)


bathrooms,1,2,3,Total
bedrooms,,,,
1.0,7928,0,0,7928
2.0,2791,0,0,2791
3.0,0,885,0,885
4.0,0,145,0,145
5.0,0,17,0,17
6.0,0,0,4,4
8.0,0,0,1,1
Total,10719,1047,5,11771


In [45]:
bedrooms = pd.to_numeric(listings_full["bedrooms"], errors="coerce")
bathrooms = pd.to_numeric(listings_full["bathrooms"], errors="coerce")

condiciones = [
    (bedrooms == 8) & (bathrooms == 3),
    (bedrooms == 6) & (bathrooms == 3),
    (bedrooms == 5) & (bathrooms == 2),
    (bedrooms == 4) & (bathrooms == 2),
    (bedrooms == 3) & (bathrooms == 2),
    bedrooms == 2,
    bedrooms == 1,
]

metros = [210, 180, 140, 120, 90, 65, 50]     #estimaciones 

listings_full["estimated_m2"] = np.select(
    condiciones,
    metros,
    default=np.nan,
)
listings_full.head(9)

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price_per_night,minimum_nights,availability_365,...,description,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,total_night_price,anual_income,estimated_m2
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,...,Perfect property if you need to come to Madrid...,2,1,1.0,2.0,5.00,28,387.0,10836.0,50.0
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,...,You'll love it. It is the ideal apartment to e...,6,1,2.0,5.0,4.98,255,414.0,105570.0,65.0
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,...,The apartment location and modern interior des...,4,1,2.0,4.0,4.95,255,213.0,54315.0,65.0
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,...,"Charming apartment with all the comforts, loca...",4,1,1.0,2.0,4.93,255,166.0,42330.0,50.0
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,...,Non-touristic stays only · Seasonal rental (ar...,6,2,3.0,4.0,4.97,255,204.0,52020.0,90.0
5,23357799,"VUT céntrico 2 habitaciones, 3 camas, 5 pers",Carabanchel,Comillas,40.392090,-3.70930,Entire home/apt,228.0,2.0,219,...,"Apartment for tourists, located 2 km from Sol ...",5,1,2.0,3.0,4.59,138,228.0,31464.0,65.0
6,23430746,Estudio en Gran Vía - Chueca,Centro,Justicia,40.422810,-3.69835,Entire home/apt,149.0,1.0,7,...,25m2 STUDIO IN THE CENTER OF MADRID (CHUECA).<...,2,1,1.0,1.0,4.98,255,149.0,37995.0,50.0
7,23451465,Private room Hortaleza ( Madrid ),Hortaleza,Pinar del Rey,40.468990,-3.65619,Private room,53.0,1.0,311,...,The room is only 4 stops away from the airport...,1,1,1.0,2.0,4.63,114,53.0,6042.0,50.0
8,23459443,Waou Sol III - Alquiler de Corta Duración,Centro,Embajadores,40.410190,-3.70436,Entire home/apt,205.0,1.0,312,...,Stunning duplex attic in the heart of Madrid. ...,6,1,2.0,2.0,4.86,24,205.0,4920.0,65.0


In [48]:
listings_full["aprox_total_cost"] = (
    listings_full["estimated_m2"]
    * listings_full["price_m2"]
    * 0.75
)
listings_full.head(5)

,id,name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price_per_night,minimum_nights,availability_365,...,accommodates,bathrooms,bedrooms,beds,review_scores_location,estimated_occupancy_l365d,total_night_price,anual_income,estimated_m2,aprox_total_cost
0,23205290,Hermosilla III by Home at Homes,Barrio de Salamanca,Recoletos,40.425290,-3.68383,Entire home/apt,387.0,7.0,83,...,2,1,1.0,2.0,5.00,28,387.0,10836.0,50.0,407437.50
1,23269747,5 Stars Short-Term Rental apartment at Plaza M...,Centro,Sol,40.414429,-3.70776,Entire home/apt,414.0,1.0,183,...,6,1,2.0,5.0,4.98,255,414.0,105570.0,65.0,378836.25
2,23287977,Great Location! CALLAO SQUARE II,Centro,Sol,40.419820,-3.70663,Entire home/apt,213.0,3.0,53,...,4,1,2.0,4.0,4.95,255,213.0,54315.0,65.0,378836.25
3,23301767,Espléndido Apartamento en Cava Baja - La Latina,Centro,Palacio,40.413430,-3.70820,Entire home/apt,166.0,2.0,312,...,4,1,1.0,2.0,4.93,255,166.0,42330.0,50.0,291412.50
4,23351387,Madrid Chueca · Renovated 2026 · 6 pax · Gran Vía,Centro,Justicia,40.420680,-3.70123,Entire home/apt,204.0,1.0,69,...,6,2,3.0,4.0,4.97,255,204.0,52020.0,90.0,524542.50
